# Diffraction reflection angles + 3D view (Bernina, `you` geometry)

Compute diffractometer angles for a given `(h,k,l)` reflection using diffcalc's
`you` UB geometry — note this beamline's convention calls the out-of-plane
detector circle **gamma**, where diffcalc itself calls it `nu`; that rename
happens automatically everywhere below.

All lab-frame directions in this applet are in **SwissFEL axes** — x
horizontal transverse, y vertical, z along the beam. diffcalc uses the same
right-handed frame relabelled cyclically (diffcalc +x, +y, +z = SwissFEL +y,
+z, +x); the conversion happens inside `bernina_prep.diffraction`, so nothing
you type or read here is in diffcalc's axes. The surface normal is an
(h,k,l) — a reciprocal-lattice direction — so it has no lab frame and is
never converted.

Fix any subset of the constraint angles (checkbox) and diffcalc solves for the
rest. Unfixed reflections may have more than one valid solution branch, shown
as columns `sol. 0`, `sol. 1`, ....

**Click a solution column** to drive the 3D view; it highlights in light blue
and the mechanism moves to those angles. **config** (the default) shows the
angle values entered on the left — e.g. chi = 90 from a preset — and follows
them as they are edited, even when no solution exists. **zero** returns every
circle to 0.

Angles are listed in one standard order — the six physical circles (gam,
delta, mu, eta, chi, phi) first, then diffcalc's virtual ones — and the
solution rows follow the same order. The **min/max** boxes beside each value
filter the solution branches; an empty box is ignored.

**load preset** applies one of the stored beamline configurations from
`apps/bernina_parameters/` — angle values, fix conditions and solution
filters only, so whatever sample and hkl you have entered are kept — and
then solves once. h, k and l accept fractional values (0.1 per spinner
click, rounded to 4 decimals).

**store config… / load config** save and restore the inputs as a JSON file.
The **file name** box sets the downloaded file's name. Groups you untick are simply left out, so a file can act as a partial
template — anything missing stays free when it is loaded back. The **3D view
display** group holds the view's own settings (display toggles, reciprocal
scale, detector distance and size); the view adds those to the file itself.

The 3D panel's own controls sit under the plot by default (uncheck *controls
under the plot* to put them beside it). Its options:

- each circle's arc shows *how far it has turned*; at 0 it falls back to the
  short arrow indicating the positive direction. A negative angle is drawn as
  the complementary 2π − |angle| arc, so the arrowhead never flips.
- **show rotation circles** hides the whole ring/arc set.
- **show angle controls** brings back the per-circle sliders (off here, since
  the solution table drives them).
- **show reciprocal vectors** draws k (ending at the sample), k&prime; (towards
  the detector, same length) and Q = k&prime; &minus; k from the sample, all
  labelled, with thin lines closing the quadrilateral.
- **show reciprocal unit cell** draws labelled a\*, b\*, c\* from the sample in
  red, green and blue, body completed in thin black lines. It rotates with
  mu/eta/chi/phi, so the tip of Q lands on the cell node of the reflection.
- **label Q with hkl + cell around it** appends the *measured* indices to the Q
  label and outlines the cell that point sits in — hkl (0.1, 2.99, 3) gives the
  cell spanning h 0–1, k 2–3, l 3–4. The indices are recomputed live in the
  view from UB and the current circle positions, so dragging any angle updates
  both the label and the cell; off a Bragg condition they are non-integer.
- **show Ewald sphere** draws the sphere of radius |k| centred on the tail of k,
  translucent blue, clipped to the view box (the view never rescales to it).
- **show detector cut of the Ewald sphere** shades in red the patch that the
  detector actually sees, from its distance and its width/height below.

Reciprocal lengths follow diffcalc's own 2π convention (|k| = 2π/λ, |Q| = 2π/d,
|a\*| = 2π/a); one slider sets how many cm of the direct-space scene represent
1 Å⁻¹ (5 cm by default).


In [ ]:
import base64
import html
import json
import sys
from pathlib import Path

try:
    import bernina_prep  # noqa: F401
except ImportError:
    sys.path.insert(0, str(Path.cwd().parents[1]))

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from bernina_prep.diffraction import (
    build_ubcalc,
    k_magnitude,
    make_constraints,
    reciprocal_basis,
    solution_angles,
    solve_hkl,
    surface_normal_phi,
)

# Narrower value fields than before, to leave room for the per-angle solution
# filters next to them; filters are half that width again.
UC_W = "150px"
FIELD_W = "98px"
FILTER_W = "49px"
HKL_W = "98px"
DESC_W = "58px"
LABEL_W = "82px"
HKL_DECIMALS = 4

LAT_NAME = "custom lattice"

# This applet speaks SwissFEL lab axes throughout: x horizontal transverse,
# y vertical, z along the beam. diffcalc uses the same right-handed frame with
# a cyclic relabelling (diffcalc +x,+y,+z = SwissFEL +y,+z,+x), and
# bernina_prep.diffraction converts at its boundary.
LAB_FRAME = "swissfel"

# One standard ordering, used for the inputs AND the solution rows: the six
# physical circles first, then diffcalc's virtual/derived angles.
PHYSICAL_ANGLES = ["gam", "delta", "mu", "eta", "chi", "phi"]
VIRTUAL_ANGLES = ["qaz", "naz", "alpha", "beta", "psi", "a_eq_b", "mu_is_gam"]
angle_names = PHYSICAL_ANGLES + VIRTUAL_ANGLES
BOOL_ANGLES = {"a_eq_b", "mu_is_gam"}

# solve_hkl reports 'gam' as 'gamma'; same order, plus the derived quantities.
ROW_ORDER = ["gamma", "delta", "mu", "eta", "chi", "phi",
             "theta", "ttheta", "qaz", "naz", "alpha", "beta", "psi",
             "tau", "betain", "betaout"]

default_fixed = {"eta", "chi", "phi"}

ctrl = {
    "uc_a": widgets.FloatText(description="a (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_b": widgets.FloatText(description="b (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_c": widgets.FloatText(description="c (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_alpha": widgets.FloatText(description="alpha", value=90, layout=widgets.Layout(width=UC_W)),
    "uc_beta": widgets.FloatText(description="beta", value=90, layout=widgets.Layout(width=UC_W)),
    "uc_gamma": widgets.FloatText(description="gamma", value=90, layout=widgets.Layout(width=UC_W)),
    # float hkl (a reflection need not sit on an integer node), stepping by 0.1
    # on the spinner and rounded to HKL_DECIMALS so the field never fills up
    # with floating-point noise
    "h": widgets.FloatText(value=1, step=0.1, description="h",
                           layout=widgets.Layout(width=HKL_W),
                           style={"description_width": "20px"}),
    "k": widgets.FloatText(value=1, step=0.1, description="k",
                           layout=widgets.Layout(width=HKL_W),
                           style={"description_width": "20px"}),
    "l": widgets.FloatText(value=1, step=0.1, description="l",
                           layout=widgets.Layout(width=HKL_W),
                           style={"description_width": "20px"}),
    # Lab-frame directions are entered in SwissFEL axes (x transverse, y up,
    # z beam) and converted to diffcalc's on the way in - see LAB_FRAME below.
    # The surface normal is an (h,k,l), so it carries no lab frame at all.
    "surface_normal": widgets.Text(value="(1,1,1)",
                                   description="surf. normal (h,k,l)",
                                   layout=widgets.Layout(width="372px"),
                                   style={"description_width": "168px"}),
    "orientations": widgets.Textarea(
        value="(1,1,1) > (1,0,0)\n(1,1,-1) > (0,1,0)",
        description="orientations (h,k,l) > (x,y,z)",
        placeholder='one "(h,k,l) > (x,y,z)" pair per line, xyz in SwissFEL axes',
        layout=widgets.Layout(width="372px", height="66px"),
        style={"description_width": "168px"},
    ),
}

# per-angle solution filters; an empty box means "no constraint"
filters = {}
angle_rows = []
for name in angle_names:
    if name in BOOL_ANGLES:
        ctrl[name] = widgets.Checkbox(value=False, indent=False,
                                      layout=widgets.Layout(width=FIELD_W))
    else:
        ctrl[name] = widgets.FloatText(value=0, layout=widgets.Layout(width=FIELD_W))
    ctrl[f"{name}_fixed"] = widgets.Checkbox(value=name in default_fixed,
                                             description="fix", indent=False,
                                             layout=widgets.Layout(width="62px"))
    row = [widgets.Label(name, layout=widgets.Layout(width=LABEL_W)), ctrl[name]]
    if name in BOOL_ANGLES:
        row.append(widgets.Label("", layout=widgets.Layout(width="106px")))
    else:
        lo = widgets.Text(placeholder="min", layout=widgets.Layout(width=FILTER_W))
        hi = widgets.Text(placeholder="max", layout=widgets.Layout(width=FILTER_W))
        filters[name] = (lo, hi)
        row += [lo, hi]
    row.append(ctrl[f"{name}_fixed"])
    angle_rows.append(widgets.HBox(row))

ctrl["energy"] = widgets.FloatSlider(value=8000, min=1000, max=20000, step=10,
                                     readout=False, continuous_update=False,
                                     layout=widgets.Layout(width="170px"))
energy_value = widgets.BoundedFloatText(description="energy", value=8000, min=1000,
                                        max=20000, step=10,
                                        layout=widgets.Layout(width="150px"),
                                        style={"description_width": DESC_W})
widgets.link((energy_value, "value"), (ctrl["energy"], "value"))
energy_row = widgets.HBox([energy_value, widgets.Label("eV"), ctrl["energy"]])

ctr_uc = widgets.HBox([ctrl[n] for n in ("uc_a", "uc_b", "uc_c", "uc_alpha", "uc_beta", "uc_gamma")],
                      layout=widgets.Layout(flex_flow="row wrap"))
ctr_hkl = widgets.HBox([ctrl["h"], ctrl["k"], ctrl["l"]],
                       layout=widgets.Layout(flex_flow="row wrap"))
ctr_or = widgets.VBox([ctrl["surface_normal"], ctrl["orientations"]])
filter_header = widgets.HTML(
    f'<div style="font-size:11px;color:#666;margin-top:6px">'
    f'<span style="display:inline-block;width:{LABEL_W}"></span>'
    f'<span style="display:inline-block;width:{FIELD_W}">value</span>'
    f'<span style="display:inline-block;width:106px">solution filter</span>'
    f'</div>')
ctr_ang = widgets.VBox([filter_header] + angle_rows)

# --- stored beamline presets ------------------------------------------------
# JSON files in apps/bernina_parameters/. These deliberately apply ONLY the
# angle values, the fix conditions and the solution filters - whatever sample
# and hkl you have entered is left untouched - and then solve once.

PRESET_DIR = Path.cwd().parent / "bernina_parameters"
PRESET_GROUPS = ("angles", "fixed", "filters")

# While a whole config is being applied, every widget write would otherwise
# fire its own solve; suspend and run exactly one at the end.
_suspend = {"on": False}


def preset_label(path):
    """Bernina_THC_diffraction_config.json -> 'Bernina THC'."""
    stem = path.stem
    for suffix in ("_diffraction_config", "_config"):
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
            break
    return stem.replace("_", " ").strip()


def find_presets():
    if not PRESET_DIR.is_dir():
        return {}
    return {preset_label(p): p for p in sorted(PRESET_DIR.glob("*.json"))}


presets = find_presets()
PRESET_NONE = "— preset —"
preset_dd = widgets.Dropdown(
    options=[PRESET_NONE] + list(presets),
    value=PRESET_NONE,
    description="load preset",
    layout=widgets.Layout(width="330px"),
    style={"description_width": "90px"},
)
preset_msg = widgets.HTML()


def on_preset(change):
    name = change["new"]
    if name == PRESET_NONE or name not in presets:
        return
    try:
        cfg = json.loads(presets[name].read_text())
    except Exception as exc:
        preset_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'
        return
    subset = {k: v for k, v in cfg.items() if k in PRESET_GROUPS}
    ignored = [k for k in cfg if k not in PRESET_GROUPS]
    _suspend["on"] = True
    try:
        apply_config(subset)
    finally:
        _suspend["on"] = False
    recompute()                      # exactly one solve after the batch
    note = f"loaded <b>{html.escape(name)}</b>: {', '.join(sorted(subset))}"
    if ignored:
        note += f" (ignored {', '.join(sorted(ignored))}; sample and hkl kept)"
    preset_msg.value = f'<span style="font-size:11px;color:#282">{note}</span>'


preset_dd.observe(on_preset, names="value")


# --- store / recall configurations ------------------------------------------

INCLUDE_GROUPS = [
    ("sample", "sample parameters (unit cell, orientations, surface normal)"),
    ("hkl", "h, k, l"),
    ("energy", "energy"),
    ("angles", "angle values"),
    ("fixed", "fix conditions"),
    ("filters", "solution filters"),
    ("display", "3D view display (toggles, reciprocal scale, detector)"),
]
include_boxes = {
    key: widgets.Checkbox(value=True, description=label, indent=False,
                          layout=widgets.Layout(width="330px"))
    for key, label in INCLUDE_GROUPS
}

store_btn = widgets.Button(description="store config…", layout=widgets.Layout(width="130px"))
make_btn = widgets.Button(description="create file", button_style="primary",
                          layout=widgets.Layout(width="110px"))
file_name = widgets.Text(value="bernina_diffraction_config", description="file name",
                         layout=widgets.Layout(width="330px"),
                         style={"description_width": "70px"})
upload = widgets.FileUpload(accept=".json", multiple=False,
                            description="load config", layout=widgets.Layout(width="150px"))
download_link = widgets.HTML()
config_msg = widgets.HTML()
store_panel = widgets.VBox(
    [widgets.HTML('<div style="font-size:11px;color:#666">include in the file '
                  '(unchecked groups are left out, so the file acts as a partial '
                  'template and anything missing stays free on load):</div>')]
    + [include_boxes[k] for k, _ in INCLUDE_GROUPS]
    + [file_name, widgets.HBox([make_btn]), download_link],
    layout=widgets.Layout(display="none", border="1px solid #ddd",
                          padding="6px", margin="4px 0 0 0"),
)


def toggle_store(_b):
    shown = store_panel.layout.display != "none"
    store_panel.layout.display = "none" if shown else "flex"


store_btn.on_click(toggle_store)


def current_config():
    cfg = {}
    if include_boxes["sample"].value:
        cfg["sample"] = {n: ctrl[n].value for n in
                         ("uc_a", "uc_b", "uc_c", "uc_alpha", "uc_beta", "uc_gamma",
                          "surface_normal", "orientations")}
    if include_boxes["hkl"].value:
        cfg["hkl"] = {n: ctrl[n].value for n in ("h", "k", "l")}
    if include_boxes["energy"].value:
        cfg["energy"] = ctrl["energy"].value
    if include_boxes["angles"].value:
        cfg["angles"] = {n: ctrl[n].value for n in angle_names}
    if include_boxes["fixed"].value:
        cfg["fixed"] = {n: ctrl[f"{n}_fixed"].value for n in angle_names}
    if include_boxes["filters"].value:
        # kept as strings, empty included on purpose: an empty filter is a
        # meaningful "free" state in a template
        cfg["filters"] = {n: [lo.value, hi.value] for n, (lo, hi) in filters.items()}
    return cfg


def config_filename():
    """The entered name, made safe for a download and ending in .json."""
    stem = file_name.value.strip()
    if stem.lower().endswith(".json"):
        stem = stem[:-5]
    stem = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in stem).strip("._")
    return (stem or "bernina_diffraction_config") + ".json"


def on_make(_b):
    cfg = current_config()
    with_display = include_boxes["display"].value
    if not cfg and not with_display:
        download_link.value = '<span style="color:#b00">nothing selected</span>'
        return
    blob = base64.b64encode(json.dumps(cfg, indent=2).encode()).decode()
    # The display settings live in the 3D page, which python cannot read: with
    # that group ticked, the link carries id + data-cfg and the page keeps its
    # href rewritten to this config plus its current display block.
    hook = (f'id="diff3d-config-download" data-cfg="{blob}" ' if with_display else "")
    name = config_filename()
    download_link.value = (
        f'<a download="{name}" {hook}'
        f'href="data:application/json;base64,{blob}" '
        f'style="font-size:12px">&#11015; download {name}</a>')


make_btn.on_click(on_make)
# a link made before renaming would still carry the old name
file_name.observe(lambda _c: setattr(download_link, "value", ""), names="value")


def apply_config(cfg):
    applied = []
    for name, value in (cfg.get("sample") or {}).items():
        if name in ctrl:
            ctrl[name].value = value
            applied.append(name)
    for name, value in (cfg.get("hkl") or {}).items():
        if name in ctrl:
            ctrl[name].value = value
            applied.append(name)
    if "energy" in cfg:
        ctrl["energy"].value = cfg["energy"]
        applied.append("energy")
    for name, value in (cfg.get("angles") or {}).items():
        if name in ctrl:
            ctrl[name].value = value
            applied.append(name)
    for name, value in (cfg.get("fixed") or {}).items():
        if f"{name}_fixed" in ctrl:
            ctrl[f"{name}_fixed"].value = bool(value)
            applied.append(f"{name}_fixed")
    for name, pair in (cfg.get("filters") or {}).items():
        if name in filters:
            filters[name][0].value = pair[0] or ""
            filters[name][1].value = pair[1] or ""
            applied.append(f"{name}_filter")
    if isinstance(cfg.get("display"), dict):
        # rides on the next push only (see push_view)
        state["display"] = cfg["display"]
        applied.append("display")
    return applied


def on_upload(change):
    items = change["new"]
    if not items:
        return
    item = items[0] if isinstance(items, (list, tuple)) else list(items.values())[0]
    try:
        raw = item["content"]
        cfg = json.loads(bytes(raw).decode())
        _suspend["on"] = True
        try:
            applied = apply_config(cfg)
        finally:
            _suspend["on"] = False
        recompute()
        config_msg.value = (f'<span style="font-size:11px;color:#282">loaded '
                            f'{len(applied)} settings; groups absent from the file '
                            f'were left as they are</span>')
    except Exception as exc:
        config_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'


upload.observe(on_upload, names="value")

config_row = widgets.VBox([preset_dd, preset_msg,
                           widgets.HBox([store_btn, upload]), store_panel, config_msg])

controls = widgets.VBox([ctr_uc, ctr_hkl, ctr_or, ctr_ang, energy_row,
                         widgets.HTML('<hr style="margin:8px 0">'), config_row],
                        layout=widgets.Layout(width="380px", flex="0 0 auto"))

# --- solutions table, one clickable column per (surviving) branch ------------

ROW_LABEL_W = 78
COL_W = 96
SELECTED_BG = "#cfe8fb"

# "selected" is a solution column index, or one of the two fixed poses below
SEL_CONFIG, SEL_ZERO = "config", "zero"
state = {"df": None, "cols": [], "selected": SEL_CONFIG, "ubcalc": None, "cons": None,
         "display": None}

solution_buttons = widgets.HBox([])
table_html = widgets.HTML()
solve_msg = widgets.HTML()
output = widgets.Output()


def _fmt(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "NaN"
    return f"{float(v):.3f}"


def _num_or_none(text):
    text = (text or "").strip()
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def passing_columns(df):
    """Columns surviving the per-angle min/max filters (empty box = ignored)."""
    keep = []
    for col in df.columns:
        ok = True
        for name, (lo_w, hi_w) in filters.items():
            row = "gamma" if name == "gam" else name
            if row not in df.index:
                continue
            lo, hi = _num_or_none(lo_w.value), _num_or_none(hi_w.value)
            if lo is None and hi is None:
                continue
            v = df.loc[row, col]
            if pd.isna(v) or (lo is not None and v < lo) or (hi is not None and v > hi):
                ok = False
                break
        if ok:
            keep.append(col)
    return keep


def render_table():
    df, cols = state["df"], state["cols"]
    if df is None or not cols:
        table_html.value = ("" if df is None else
                            '<span style="font-size:12px;color:#a60">'
                            'no solution passes the filters</span>')
        return
    sel = state["selected"]
    head = "".join(
        f'<th style="width:{COL_W}px;text-align:right;padding:2px 6px;'
        f'border-bottom:1px solid #ccc;'
        f'{"background:" + SELECTED_BG + ";" if i == sel else ""}">{html.escape(str(c))}</th>'
        for i, c in enumerate(cols))
    body = []
    for name in df.index:
        cells = "".join(
            f'<td style="width:{COL_W}px;text-align:right;padding:1px 6px;'
            f'font-variant-numeric:tabular-nums;'
            f'{"background:" + SELECTED_BG + ";" if i == sel else ""}">'
            f'{_fmt(df.loc[name, c])}</td>'
            for i, c in enumerate(cols))
        body.append(
            f'<tr><td style="width:{ROW_LABEL_W}px;text-align:right;padding:1px 6px;'
            f'font-weight:600;">{html.escape(str(name))}</td>{cells}</tr>')
    table_html.value = ('<table style="border-collapse:collapse;font-size:12px;">'
                        f'<tr><td style="width:{ROW_LABEL_W}px"></td>{head}</tr>'
                        + "".join(body) + "</table>")


def select_solution(index):
    state["selected"] = index
    for b in solution_buttons.children:
        b.button_style = "info" if b._sel == index else ""
    render_table()
    push_view()


def rebuild_solution_buttons():
    buttons = []
    for i, col in enumerate(state["cols"]):
        b = widgets.Button(description=str(col), tooltip=f"show {col} in the 3D view",
                           layout=widgets.Layout(width=f"{COL_W}px", margin="0"))
        b.on_click(lambda _b, idx=i: select_solution(idx))
        b._sel = i
        buttons.append(b)
    # config sits in the row-label slot so the solution buttons stay aligned
    # with their table columns; zero goes after them
    config = widgets.Button(description="config",
                            tooltip="show the angle values entered on the left",
                            layout=widgets.Layout(width=f"{ROW_LABEL_W}px", margin="0"))
    config.on_click(lambda _b: select_solution(SEL_CONFIG))
    config._sel = SEL_CONFIG
    zero = widgets.Button(description="zero", tooltip="move all circles back to 0",
                          layout=widgets.Layout(width="60px", margin="0 0 0 8px"))
    zero.on_click(lambda _b: select_solution(SEL_ZERO))
    zero._sel = SEL_ZERO
    solution_buttons.children = [config] + buttons + [zero]


# --- the 3D view ------------------------------------------------------------

_html_path = Path.cwd() / "diffractometer_3d.html"
_scene_html = _html_path.read_text()

view_frame = widgets.HTML(
    f'<iframe id="diff3d" srcdoc="{html.escape(_scene_html)}" '
    'style="width:100%; height:760px; border:1px solid #ddd;"></iframe>',
    layout=widgets.Layout(width="100%"),
)
payload_holder = widgets.HTML()
_push_count = {"n": 0}


def config_angles():
    """The six physical circles as entered on the left (view naming: gamma)."""
    return {("gamma" if n == "gam" else n): float(ctrl[n].value) for n in PHYSICAL_ANGLES}


def push_view():
    sel = state["selected"]
    angles = {n: 0.0 for n in ("mu", "eta", "chi", "phi", "delta", "gamma")}
    label = ""
    if sel == SEL_CONFIG:
        angles, label = config_angles(), "config"
    elif isinstance(sel, int) and state["df"] is not None and sel < len(state["cols"]):
        col = state["cols"][sel]
        angles = solution_angles(state["df"], col)
        label = str(col)
    msg = {"angles": angles, "kMag": k_magnitude(ctrl["energy"].value),
           "label": label, "n": _push_count["n"]}
    if state["ubcalc"] is not None:
        # the view derives the measured hkl itself, live, from UB + the angles
        msg["ub"] = reciprocal_basis(state["ubcalc"])
        # so the sample platelet shows the surface orientation actually entered
        msg["nPhi"] = surface_normal_phi(state["ubcalc"])
    if state["display"] is not None:
        # sent once, so the view's own toggles are not reset on every push
        msg["display"] = state["display"]
        state["display"] = None
    _push_count["n"] += 1
    payload_holder.value = ('<span id="diff3d-payload" style="display:none">'
                            + html.escape(json.dumps(msg)) + "</span>")


# --- solve ------------------------------------------------------------------

def recompute(*_):
    global last_ubcalc, last_cons
    if _suspend["on"]:
        return
    ubcalc = None
    try:
        ubcalc = build_ubcalc(
            LAT_NAME,
            ctrl["uc_a"].value, ctrl["uc_b"].value, ctrl["uc_c"].value,
            ctrl["uc_alpha"].value, ctrl["uc_beta"].value, ctrl["uc_gamma"].value,
            [ln.strip() for ln in ctrl["orientations"].value.splitlines() if ln.strip()],
            ctrl["surface_normal"].value,
            frame=LAB_FRAME,
        )
        fixed = {n: ctrl[n].value for n in angle_names if ctrl[f"{n}_fixed"].value}
        cons = make_constraints(fixed)
        df = solve_hkl(ubcalc, cons, ctrl["h"].value, ctrl["k"].value,
                       ctrl["l"].value, ctrl["energy"].value)
    except Exception as exc:
        solve_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'
        state.update(df=None, cols=[])
        # a sample can be perfectly valid while this hkl/constraint set has no
        # solution; the view must still show the new cell, not the old one
        if ubcalc is not None:
            state["ubcalc"] = ubcalc
        if isinstance(state["selected"], int):
            state["selected"] = SEL_CONFIG
        rebuild_solution_buttons()
        select_solution(state["selected"])
        return

    # one standard row order, matching the input ordering
    ordered = [r for r in ROW_ORDER if r in df.index] + \
              [r for r in df.index if r not in ROW_ORDER]
    df = df.reindex(ordered)

    solve_msg.value = ""
    cols = passing_columns(df)
    state.update(df=df, cols=cols, ubcalc=ubcalc, cons=cons)
    last_ubcalc, last_cons = ubcalc, cons
    if isinstance(state["selected"], int) and state["selected"] >= len(cols):
        state["selected"] = SEL_CONFIG
    rebuild_solution_buttons()
    select_solution(state["selected"])


def _round_hkl(change):
    """Keep h, k, l to HKL_DECIMALS places.

    The correction is applied with solving suspended, so a rounded edit still
    costs exactly one solve rather than two.
    """
    rounded = round(float(change["new"]), HKL_DECIMALS)
    if rounded != change["new"]:
        _suspend["on"] = True
        try:
            change["owner"].value = rounded
        finally:
            _suspend["on"] = False


for _n in ("h", "k", "l"):
    ctrl[_n].observe(_round_hkl, names="value")

for _w in ctrl.values():
    _w.observe(recompute, names="value")
for _lo, _hi in filters.values():
    _lo.observe(recompute, names="value")
    _hi.observe(recompute, names="value")
recompute()

solutions_panel = widgets.VBox(
    [widgets.HTML("<b>solutions</b> &mdash; click a column to show it in the 3D view"),
     solve_msg, solution_buttons, table_html],
    layout=widgets.Layout(flex="0 0 auto", margin="0 0 0 12px"),
)
view_panel = widgets.VBox(
    [view_frame, payload_holder],
    layout=widgets.Layout(flex="1 1 520px", min_width="0", margin="0 0 0 12px"),
)

display(widgets.HBox(
    [controls, solutions_panel, view_panel],
    layout=widgets.Layout(flex_flow="row wrap", width="100%", align_items="flex-start"),
))


## Table of reflections

Reuses the crystal/orientation and fixed constraints set above. Enter one
`h,k,l` per line and pick which solution branch to report when a reflection
has more than one; reflections diffcalc can't solve are flagged in the
`error` column rather than aborting the whole table.

In [ ]:
from bernina_prep.diffraction import reflection_table

hkl_text = widgets.Textarea(
    value="1,0,0\n0,1,0\n0,0,1\n1,1,1",
    description="h,k,l list",
    layout=widgets.Layout(width="200px", height="120px"),
)
solution_idx = widgets.IntText(value=0, description="solution #")
compute_btn = widgets.Button(description="Compute table", button_style="primary")
table_out = widgets.Output()


def parse_hkl_lines(text):
    reflections = []
    for line in text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        h, k, l = (float(x) for x in line.replace(";", ",").split(","))
        reflections.append((h, k, l))
    return reflections


def on_compute(_):
    table_out.clear_output()
    with table_out:
        if "last_ubcalc" not in globals():
            print("Set up the crystal/orientation above first (run the single-reflection cell).")
            return
        reflections = parse_hkl_lines(hkl_text.value)
        display(reflection_table(last_ubcalc, last_cons, reflections, ctrl["energy"].value, solution=solution_idx.value))


compute_btn.on_click(on_compute)
display(widgets.VBox([widgets.HBox([hkl_text, solution_idx]), compute_btn, table_out]))


## Detector footprint (Q / hkl range)

For the currently-set crystal/orientation and a chosen `(h,k,l)`/solution
branch above, computes the range of reciprocal-lattice points a flat area
detector actually covers, given its distance from the sample (`tdet`) and
its physical size. Uses diffcalc's own `q_phi` convention directly (not a
second geometry library), extended from the single central ray to the
whole detector face, so it's consistent with the angles calculated above.

In [ ]:
from bernina_prep.diffraction import solve_hkl_positions, detector_hkl_grid, detector_hkl_summary

tdet_w = widgets.FloatText(value=500.0, description="tdet [mm]")
det_w_w = widgets.FloatText(value=77.0, description="width [mm]")
det_h_w = widgets.FloatText(value=77.0, description="height [mm]")
grid_n_w = widgets.IntText(value=9, description="grid n")
footprint_solution_idx = widgets.IntText(value=0, description="solution #")
footprint_btn = widgets.Button(description="Compute Q range", button_style="primary")
plot_btn = widgets.Button(description="Show 3D plot (slower)", button_style="")
footprint_out = widgets.Output()
plot_out = widgets.Output()

_last_footprint = {}


def _current_position():
    h, k, l = ctrl["h"].value, ctrl["k"].value, ctrl["l"].value
    energy = ctrl["energy"].value
    result = solve_hkl_positions(last_ubcalc, last_cons, h, k, l, energy)
    idx = footprint_solution_idx.value if footprint_solution_idx.value < len(result) else len(result) - 1
    pos, _ = result[idx]
    return pos, energy


def on_footprint(_):
    footprint_out.clear_output()
    with footprint_out:
        if "last_ubcalc" not in globals():
            print("Set up the crystal/orientation above first (run the single-reflection cell).")
            return
        pos, energy = _current_position()
        H, K, L = detector_hkl_grid(
            last_ubcalc, pos, energy, tdet_w.value, det_w_w.value, det_h_w.value, grid_n_w.value
        )
        _last_footprint.update(pos=pos, energy=energy, H=H, K=K, L=L)
        summary = detector_hkl_summary(
            last_ubcalc, pos, energy, tdet_w.value, det_w_w.value, det_h_w.value, grid_n_w.value
        )
        print(f"Detector center at delta={pos.delta:.3f}, gamma={pos.nu:.3f}, tdet={tdet_w.value:.1f} mm")
        display(summary.to_frame("value"))


def on_plot(_):
    plot_out.clear_output()
    with plot_out:
        if not _last_footprint:
            print("Compute the Q range first.")
            return
        import matplotlib.pyplot as plt

        H, K, L = _last_footprint["H"], _last_footprint["K"], _last_footprint["L"]
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(projection="3d")
        ax.plot_surface(H, K, L, alpha=0.6)
        ax.set_xlabel("h")
        ax.set_ylabel("k")
        ax.set_zlabel("l")
        n = H.shape[0] // 2
        cx, cy, cz = H[n, n], K[n, n], L[n, n]
        arrowlen = 0.3 * max(H.max() - H.min(), K.max() - K.min(), L.max() - L.min(), 1e-6)
        ax.quiver(cx, cy, cz, arrowlen, 0, 0, color="r")
        ax.quiver(cx, cy, cz, 0, arrowlen, 0, color="g")
        ax.quiver(cx, cy, cz, 0, 0, arrowlen, color="b")
        plt.show()


footprint_btn.on_click(on_footprint)
plot_btn.on_click(on_plot)
display(
    widgets.VBox(
        [
            widgets.HBox([tdet_w, det_w_w, det_h_w, grid_n_w, footprint_solution_idx]),
            widgets.HBox([footprint_btn, plot_btn]),
            footprint_out,
            plot_out,
        ]
    )
)
